## Creation of Graz dataset

In [13]:
import pandas as pd

In [20]:
def process_graz_csv(df_raw):
    df_raw["time"] = pd.to_datetime(df_raw["time"], format='mixed')
    df_raw["hour"] = df_raw["time"].dt.hour
    #df = df_raw[(df_raw["count"] > 0) & (df_raw["time"].dt.hour == 8)]
    df = df_raw[(df_raw["count"] > 0)]
    df['id'] = df['Name'].str.extract(r'^[^_]+_[^_]+_([^\.]+)(?:\..+)?$')
    grouped_df = df[["id","count","time"]].groupby(["id","time"], as_index=False).sum()
    return grouped_df


In [21]:
file_1 = "stations_datasets/raw/Graz/Q1_Detektor-Messquerschnitt-Zählung-250101_250331.csv"
file_2 = "stations_datasets/raw/Graz/Q2_Detektor-Messquerschnitt-Zählung-250401_250630.csv"
file_3 = "stations_datasets/raw/Graz/Q3_Detektor-Messquerschnitt-Zählung-250701_250930.csv"
file_4 = "stations_datasets/raw/Graz/Q4_Detektor-Messquerschnitt-Zählung-251001_251231.csv"


df_1_raw = pd.read_csv(file_1, decimal=',',
                    usecols=[0, 1, 2]).rename(columns = {"Q@KFZ": "count", "Time":"time"})

df_2_raw = pd.read_csv(file_2, decimal=',', sep = ";").rename(columns = {"Q@KFZ": "count", "Time":"time"})

df_3_raw = pd.read_csv(file_3, decimal=',', sep = ";").rename(columns = {"Q@KFZ": "count", "Time":"time"})

df_4_raw = pd.read_csv(file_4, decimal=',', sep = ";").rename(columns = {"Q@KFZ": "count", "Time":"time"})



df_1 = process_graz_csv(df_1_raw)
df_2 = process_graz_csv(df_2_raw)
df_3 = process_graz_csv(df_3_raw)
df_4 = process_graz_csv(df_4_raw)



In [22]:
final_df_graz = pd.concat([df_1,df_2,df_3,df_4], axis=0)
final_df_graz["id"] = "GRZ_" + final_df_graz["id"]
final_df_graz["fetched_city"] = 'Graz'
final_df_graz

,id,time,count,fetched_city
0,GRZ_017,2025-01-01 00:59:00,2.000000,Graz
1,GRZ_017,2025-01-01 03:00:00,1.000000,Graz
2,GRZ_017,2025-01-01 03:59:00,1.000000,Graz
3,GRZ_017,2025-01-01 05:00:00,1.000000,Graz
4,GRZ_017,2025-01-01 06:00:00,1.000000,Graz
...,...,...,...,...
34383,GRZ_D33104,2025-12-31 20:00:00,8.983334,Graz
34384,GRZ_D33104,2025-12-31 21:00:00,2.950000,Graz
34385,GRZ_D33104,2025-12-31 22:00:00,12.983334,Graz
34386,GRZ_D33104,2025-12-31 23:00:00,4.000000,Graz


In [23]:
final_df_graz.to_csv("stations_datasets/processed/graz_stations_count.csv", index=False)

#### Load GeoJSON

In [107]:
import geopandas as gpd

gdf = gpd.read_file("stations_datasets/raw/Graz/graz_stations.geojson")
gdf = gdf[~gdf["count_id"].isnull()][["count_id","geometry"]].rename(columns={"count_id":"id","geometry":"geom"})
gdf["id"] = "GRZ_" + gdf["id"]
gdf["fetched_city"] = 'Graz'
gdf.to_file("stations_datasets/processed/graz_stations_ref.json", index=False)

In [ ]:
# import os
# from sqlalchemy import create_engine
# db_url = (
#     f"postgresql://{os.getenv('POSTGRES_USER')}:"
#     f"{os.getenv('POSTGRES_PASSWORD')}@"
#     f"localhost:5432/"
#     f"{os.getenv('POSTGRES_DB')}"
# )

# engine = create_engine(db_url)

# gdf.to_postgis("graz_stations", engine)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from prepare_validation import add_station_dataset

path_stations_count = "stations_datasets/processed/graz_stations_count.csv"
path_stations_ref = "stations_datasets/processed/graz_stations_ref.json"
add_station_dataset(counting_stations_count_path=path_stations_count,counting_stations_ref_path=path_stations_ref,
                    counting_stations_count_table="counting_stations_count",counting_stations_ref_table="counting_stations_ref", 
                    city_name = 'Graz')

#### Metrics Calculation

In [6]:
merged_graz_df = pd.merge(final_df_graz, gdf[["count_id","geometry"]], how="inner", left_on="id", right_on="count_id")
merged_graz_df

,id,Time,count,count_id,geometry
0,017,2025-01-01 08:00:00,1.000000,017,POINT (15.48796 47.10325)
1,017,2025-01-02 08:00:00,2.000000,017,POINT (15.48796 47.10325)
2,017,2025-01-03 08:00:00,6.000000,017,POINT (15.48796 47.10325)
3,017,2025-01-13 08:00:00,43.000000,017,POINT (15.48796 47.10325)
4,017,2025-01-14 08:00:00,41.016666,017,POINT (15.48796 47.10325)
...,...,...,...,...,...
5948,922,2025-12-27 08:00:00,2.115833,922,POINT (15.41252 47.06369)
5949,922,2025-12-30 08:00:00,6.000000,922,POINT (15.41218 47.06368)
5950,922,2025-12-30 08:00:00,6.000000,922,POINT (15.41252 47.06369)
5951,922,2025-12-31 08:00:00,1.000000,922,POINT (15.41218 47.06368)


In [29]:
test_df = merged_graz_df[merged_graz_df["count_id"] == '558']
test_df

,id,Time,count,count_id,geometry
635,558,2025-01-01 08:00:00,42.983334,558,POINT (15.45715 47.06287)
636,558,2025-01-02 08:00:00,120.666667,558,POINT (15.45715 47.06287)
637,558,2025-01-03 08:00:00,124.966667,558,POINT (15.45715 47.06287)
638,558,2025-01-13 08:00:00,731.850006,558,POINT (15.45715 47.06287)
639,558,2025-01-14 08:00:00,773.655556,558,POINT (15.45715 47.06287)
...,...,...,...,...,...
5094,558,2025-12-27 08:00:00,99.949998,558,POINT (15.45715 47.06287)
5095,558,2025-12-28 08:00:00,55.966667,558,POINT (15.45715 47.06287)
5096,558,2025-12-29 08:00:00,262.316665,558,POINT (15.45715 47.06287)
5097,558,2025-12-30 08:00:00,230.650284,558,POINT (15.45715 47.06287)


In [30]:
test_df["count"].mean()

np.float64(580.1644310082193)

In [18]:
year_graz_df = merged_graz_df[["count_id", "count"]].groupby(by="count_id", as_index=False).mean()
year_graz_df

,count_id,count
0,017,30.103397
1,102,72.245617
2,103,36.747056
3,1091,156.427317
4,1092,67.518085
5,1116,43.590624
6,1152,11.595912
7,390,100.163208
8,501,111.450954
9,558,580.164431


In [11]:
query = """SELECT p.count_id as counter_id, SUM(l.count_occ) as count_sim
FROM graz_stations p
JOIN (SELECT * FROM final_table WHERE fetched_city = 'Graz') l
ON ST_Distance(p.geometry::geography, l.line_geom::geography) < 0.1
WHERE p.count_id IS NOT NULL
GROUP BY p.count_id ;"""

sim_graz_df = pd.read_sql(query,engine)
sim_graz_df

,counter_id,count_sim
0,017,6468
1,102,3446
2,103,1229
3,104,1229
4,1091,7733
5,1092,7170
6,1116,3268
7,390,8495
8,501,7483
9,558,971


In [24]:
year_graz_df = year_graz_df[~year_graz_df["count_id"].isin(['558','749'])]

In [25]:
year_graz_df["rank"] = year_graz_df["count"].rank(ascending=False)
sim_graz_df["rank_sim"] = sim_graz_df["count_sim"].rank(ascending=False)
final_result_df = pd.merge(year_graz_df,sim_graz_df, left_on ="count_id",right_on="counter_id")
final_result_df["rank_diff"] = final_result_df["rank"] - final_result_df["rank_sim"]
final_result_df

,count_id,count,rank,counter_id,count_sim,rank_sim,rank_diff
0,017,30.103397,10.0,017,6468,6.0,4.0
1,102,72.245617,6.0,102,3446,9.0,-3.0
2,103,36.747056,9.0,103,1229,14.5,-5.5
3,1091,156.427317,3.0,1091,7733,3.0,0.0
4,1092,67.518085,7.0,1092,7170,5.0,2.0
5,1116,43.590624,8.0,1116,3268,10.0,-2.0
6,390,100.163208,5.0,390,8495,2.0,3.0
7,501,111.450954,4.0,501,7483,4.0,0.0
8,759,8.483888,13.0,759,1601,12.0,1.0
9,767,7.571142,14.0,767,1127,16.0,-2.0


In [26]:
final_result_df[["rank","rank_sim"]].corr()

,rank,rank_sim
rank,1.000000,0.846502
rank_sim,0.846502,1.000000


In [27]:
from scipy.stats import spearmanr

res = spearmanr(final_result_df["rank"], final_result_df["rank_sim"])

print("Spearman correlation:", res.statistic)
print("p-value:", res.pvalue)

Spearman correlation: 0.8881118881118882
p-value: 0.00011413359814618021


### Graz Results

In [2]:
%load_ext autoreload
%autoreload 2
from prepare_validation import add_station_dataset

path_stations_count = "stations_datasets/processed/graz_stations_count.csv"
path_stations_ref = "stations_datasets/processed/graz_stations_ref.json"
input_query = "SELECT * FROM temp_hubs_final_graz_no_zones"
add_station_dataset(input_query,counting_stations_count_path=path_stations_count,counting_stations_ref_path=path_stations_ref,
                    counting_stations_count_table="counting_stations_count",counting_stations_ref_table="counting_stations_ref", 
                    city_name = 'Graz')

In [3]:
import os
from sqlalchemy import create_engine
db_url = (
    f"postgresql://{os.getenv('POSTGRES_USER')}:"
    f"{os.getenv('POSTGRES_PASSWORD')}@"
    f"localhost:5432/"
    f"{os.getenv('POSTGRES_DB')}"
)

engine = create_engine(db_url)


In [4]:
import pandas as pd
query = """SELECT * FROM counting_stations_count WHERE fetched_city = 'Graz'"""
graz_df = pd.read_sql(query,engine)
graz_df

,id,time,count,count_sim,fetched_city
0,GRZ_103,2025-01-01 00:59:00,2.983333,1499,Graz
1,GRZ_103,2025-01-01 02:00:00,12.966666,1499,Graz
2,GRZ_103,2025-01-01 03:00:00,2.000000,1499,Graz
3,GRZ_103,2025-01-01 03:59:00,3.966667,1499,Graz
4,GRZ_103,2025-01-01 05:00:00,1.000000,1499,Graz
...,...,...,...,...,...
89389,GRZ_922,2025-12-31 15:00:00,2.000000,3519,Graz
89390,GRZ_922,2025-12-31 16:00:00,5.000000,3519,Graz
89391,GRZ_922,2025-12-31 17:00:00,2.000000,3519,Graz
89392,GRZ_922,2025-12-31 19:00:00,2.000000,3519,Graz


In [5]:
graz_df["date"] = pd.to_datetime(graz_df["time"]).dt.date
daily_df = graz_df[["id","date","count","count_sim"]].groupby(["id","date","count_sim"],as_index=False).sum()
num_days_df = daily_df[["id","date"]].groupby(["id"],as_index=False).count().rename(columns={"date":"num_days"})
total_count_df = daily_df[["id","count","count_sim"]].groupby(["id","count_sim"],as_index=False).sum()
average_daily_df = pd.merge(total_count_df,num_days_df)
average_daily_df["count_aadt"] = average_daily_df["count"] / average_daily_df["num_days"]
average_daily_df 

,id,count_sim,count,num_days,count_aadt
0,GRZ_102,5500,4.363644e+05,267,1634.323467
1,GRZ_103,1499,2.530088e+05,362,698.919419
2,GRZ_1091,7170,4.146141e+04,28,1480.764537
3,GRZ_1092,803,3.607671e+05,365,988.402904
4,GRZ_1116,3063,1.856256e+05,313,593.053149
5,GRZ_1152,164,1.442596e+05,366,394.151798
6,GRZ_390,2322,3.505091e+05,366,957.675031
7,GRZ_501,6008,6.288633e+05,366,1718.205615
8,GRZ_558,6771,2.951030e+06,366,8062.923561
9,GRZ_569,3713,7.661069e+05,364,2104.689173


In [6]:
grouped_graz_df = average_daily_df[["id", "count_aadt","count_sim"]].groupby("id", as_index=False).sum()
grouped_graz_df

,id,count_aadt,count_sim
0,GRZ_102,1634.323467,5500
1,GRZ_103,698.919419,1499
2,GRZ_1091,1480.764537,7170
3,GRZ_1092,988.402904,803
4,GRZ_1116,593.053149,3063
5,GRZ_1152,394.151798,164
6,GRZ_390,957.675031,2322
7,GRZ_501,1718.205615,6008
8,GRZ_558,8062.923561,6771
9,GRZ_569,2104.689173,3713


In [9]:
#grouped_graz_df_clean = grouped_graz_df[~grouped_graz_df["id"].isin(['GRZ_749'])]
grouped_graz_df_clean = grouped_graz_df
grouped_graz_df_clean["rank"] = grouped_graz_df_clean["count_aadt"].rank(ascending=False)
grouped_graz_df_clean["rank_sim"] = grouped_graz_df_clean["count_sim"].rank(ascending=False)
grouped_graz_df_clean["rank_diff"] = grouped_graz_df_clean["rank"] - grouped_graz_df_clean["rank_sim"]

grouped_graz_df_clean

,id,count_aadt,count_sim,rank,rank_sim,rank_diff
0,GRZ_102,1634.323467,5500,6.0,5.0,1.0
1,GRZ_103,698.919419,1499,10.0,11.0,-1.0
2,GRZ_1091,1480.764537,7170,7.0,2.0,5.0
3,GRZ_1092,988.402904,803,8.0,12.0,-4.0
4,GRZ_1116,593.053149,3063,11.0,8.0,3.0
5,GRZ_1152,394.151798,164,12.0,13.0,-1.0
6,GRZ_390,957.675031,2322,9.0,10.0,-1.0
7,GRZ_501,1718.205615,6008,5.0,4.0,1.0
8,GRZ_558,8062.923561,6771,2.0,3.0,-1.0
9,GRZ_569,2104.689173,3713,4.0,6.0,-2.0


In [10]:
from scipy.stats import pearsonr
from scipy.stats import spearmanr

res = spearmanr(grouped_graz_df_clean["rank"], grouped_graz_df_clean["rank_sim"])

print("Spearman correlation:", res.statistic)
print("p-value:", res.pvalue)

Spearman correlation: 0.5494505494505494
p-value: 0.05177062555916278


In [19]:
from scipy.stats import kendalltau
res = kendalltau(grouped_graz_df_clean["rank"], grouped_graz_df_clean["rank_sim"])

print("kendalltau correlation:", res.statistic)
print("p-value:", res.pvalue)

kendalltau correlation: 0.6363636363636362
p-value: 0.003181646992410881


In [20]:
from scipy.stats import pearsonr
res = pearsonr(grouped_graz_df_clean["count_aadt"], grouped_graz_df_clean["count_sim"])

print("Pearson correlation:", res.statistic)
print("p-value:", res.pvalue)

Pearson correlation: 0.872738754474632
p-value: 0.00021150451316839664


In [54]:
from scipy.stats import pearsonr
res = spearmanr(grouped_graz_df_clean_sim["rank"], grouped_graz_df_clean_sim["rank_sim"])

print("Spearman correlation:", res.statistic)
print("p-value:", res.pvalue)

Spearman correlation: 0.7342657342657343
p-value: 0.006543490146837884


In [49]:
from scipy.stats import pearsonr
res = pearsonr(grouped_graz_df_clean_sim["count_aadt"], grouped_graz_df_clean_sim["count_sim"])

print("Pearson correlation:", res.statistic)
print("p-value:", res.pvalue)

Pearson correlation: 0.5828675377225832
p-value: 0.046701121890657914


In [59]:
from scipy.stats import kendalltau
res = kendalltau(grouped_graz_df_clean_sim["rank"], grouped_graz_df_clean_sim["rank_sim"])

print("kendalltau correlation:", res.statistic)
print("p-value:", res.pvalue)

kendalltau correlation: 0.5151515151515151
p-value: 0.020980351631393297


In [5]:
import pandas as pd
import geopandas as gpd
zones_df = pd.read_csv("/home/jmartinez/Documents/bike_project/graz_poidpy.csv")
zones_df['geometry'] = gpd.GeoSeries.from_wkt(zones_df['geometry'])
zones_gdf = gpd.GeoDataFrame(zones_df, geometry='geometry', crs=4326)
zones_gdf.to_postgis("poidpy_graz_results",engine)

### Berlin validation

In [13]:
from prepare_validation import add_station_dataset

path_stations_count = "stations_datasets/processed/berlin_stations_count.csv"
path_stations_ref = "stations_datasets/processed/berlin_stations_ref_clean.geojson"
input_query = "SELECT * FROM final_table WHERE fetched_city = 'Berlin'"
add_station_dataset(input_query,counting_stations_count_path=path_stations_count,counting_stations_ref_path=path_stations_ref,
                    counting_stations_count_table="counting_stations_count",counting_stations_ref_table="counting_stations_ref", 
                    city_name = 'Berlin')

In [14]:
import os
from sqlalchemy import create_engine
db_url = (
    f"postgresql://{os.getenv('POSTGRES_USER')}:"
    f"{os.getenv('POSTGRES_PASSWORD')}@"
    f"localhost:5432/"
    f"{os.getenv('POSTGRES_DB')}"
)

engine = create_engine(db_url)


In [15]:
import pandas as pd
query = """SELECT * FROM counting_stations_count WHERE fetched_city = 'Berlin'"""
berlin_df = pd.read_sql(query,engine)
berlin_df

,id,time,count,count_sim,fetched_city
0,BER-01-MI-AL-W,2024-01-01 00:00:00,5.0,1465,Berlin
1,BER-01-MI-AL-W,2024-01-01 01:00:00,11.0,1465,Berlin
2,BER-01-MI-AL-W,2024-01-01 02:00:00,15.0,1465,Berlin
3,BER-01-MI-AL-W,2024-01-01 03:00:00,13.0,1465,Berlin
4,BER-01-MI-AL-W,2024-01-01 04:00:00,5.0,1465,Berlin
...,...,...,...,...,...
219595,BER-27-RE-MAR,2024-12-31 19:00:00,13.0,860,Berlin
219596,BER-27-RE-MAR,2024-12-31 20:00:00,7.0,860,Berlin
219597,BER-27-RE-MAR,2024-12-31 21:00:00,5.0,860,Berlin
219598,BER-27-RE-MAR,2024-12-31 22:00:00,7.0,860,Berlin


In [16]:
berlin_df["date"] = pd.to_datetime(berlin_df["time"]).dt.date
daily_df = berlin_df[["id","date","count","count_sim"]].groupby(["id","date","count_sim"],as_index=False).sum()
num_days_df = daily_df[["id","date"]].groupby(["id"],as_index=False).count().rename(columns={"date":"num_days"})
total_count_df = daily_df[["id","count","count_sim"]].groupby(["id","count_sim"],as_index=False).sum()
average_daily_df = pd.merge(total_count_df,num_days_df)
average_daily_df["count_aadt"] = average_daily_df["count"] / average_daily_df["num_days"]
average_daily_df 

,id,count_sim,count,num_days,count_aadt
0,BER-01-MI-AL-W,1465,1060104.0,366,2896.459016
1,BER-02-MI-JAN-N,1189,1240461.0,366,3389.237705
2,BER-02-MI-JAN-S,1123,647268.0,366,1768.491803
3,BER-03-MI-SAN-O,1346,821943.0,366,2245.745902
4,BER-03-MI-SAN-W,1487,828365.0,366,2263.292350
5,BER-04-MI-NO,93,453831.0,366,1239.975410
6,BER-05-FK-OBB-O,2959,2004768.0,366,5477.508197
7,BER-05-FK-OBB-W,3313,1936945.0,366,5292.199454
8,BER-06-FK-FRA-O,898,904467.0,366,2471.221311
9,BER-06-FK-FRA-W,968,803507.0,366,2195.374317


In [21]:
grouped_berlin_df = average_daily_df[["id","count_aadt","count_sim"]]
grouped_berlin_df

,id,count_aadt,count_sim
0,BER-01-MI-AL-W,2896.459016,1465
1,BER-02-MI-JAN-N,3389.237705,1189
2,BER-02-MI-JAN-S,1768.491803,1123
3,BER-03-MI-SAN-O,2245.745902,1346
4,BER-03-MI-SAN-W,2263.292350,1487
5,BER-04-MI-NO,1239.975410,93
6,BER-05-FK-OBB-O,5477.508197,2959
7,BER-05-FK-OBB-W,5292.199454,3313
8,BER-06-FK-FRA-O,2471.221311,898
9,BER-06-FK-FRA-W,2195.374317,968


In [23]:
#grouped_berlin_df = grouped_berlin_df[~grouped_berlin_df["id"].isin(['GRZ_749'])]

grouped_berlin_df["rank"] = grouped_berlin_df["count_aadt"].rank(ascending=False)
grouped_berlin_df["rank_sim"] = grouped_berlin_df["count_sim"].rank(ascending=False)
grouped_berlin_df["rank_diff"] = grouped_berlin_df["rank"] - grouped_berlin_df["rank_sim"]

grouped_berlin_df

,id,count_aadt,count_sim,rank,rank_sim,rank_diff
0,BER-01-MI-AL-W,2896.459016,1465,7.0,7.0,0.0
1,BER-02-MI-JAN-N,3389.237705,1189,4.0,9.0,-5.0
2,BER-02-MI-JAN-S,1768.491803,1123,22.0,10.0,12.0
3,BER-03-MI-SAN-O,2245.745902,1346,17.0,8.0,9.0
4,BER-03-MI-SAN-W,2263.292350,1487,16.0,6.0,10.0
5,BER-04-MI-NO,1239.975410,93,25.0,24.0,1.0
6,BER-05-FK-OBB-O,5477.508197,2959,1.0,2.0,-1.0
7,BER-05-FK-OBB-W,5292.199454,3313,2.0,1.0,1.0
8,BER-06-FK-FRA-O,2471.221311,898,14.0,15.0,-1.0
9,BER-06-FK-FRA-W,2195.374317,968,18.0,13.0,5.0


In [25]:
from scipy.stats import spearmanr
from scipy.stats import pearsonr

res = pearsonr(grouped_berlin_df["count_aadt"], grouped_berlin_df["count_sim"])

print("Spearman correlation:", res.statistic)
print("p-value:", res.pvalue)

Spearman correlation: 0.5276745383670947
p-value: 0.00671040159340709
